In [1]:
import pandas as pd
import numpy as np

In [2]:
books = pd.read_parquet("../data/books-final.parquet")
books

,title,subtitle,title_and_subtitle,authors,description,language,isbn13s,tags,year,bx_popularity,gr_popularity,description_source,desc_class
work_id,,,,,,,,,,,,,
OL100003W,Home fires,NaN,Home fires,[Margaret Maron],North Carolina judge Deborah Knott investigate...,en,[9780446608107],"[Arson, Detective And Mystery Stories, Mystery...",2000,8,0,bookcrossing,unique
OL1000053W,No Marriage of Convenience (Avon Romantic Trea...,NaN,No Marriage of Convenience (Avon Romantic Trea...,[Elizabeth Boyle],"Mason St. Clair, the new Earl of Ashlin, has i...",en,[9780380815340],"[Fiction, History, General, Romance]",2000,8,0,bookcrossing,unique
OL1000055W,Once Tempted,NaN,Once Tempted,[Elizabeth Boyle],"Innocence LostA notorious rake, the Marquis of...",en,[9780380815357],"[Spain, Romance, Fiction, General, Man-Woman R...",2001,7,0,bookcrossing,unique
OL1000062W,It takes a hero,NaN,It takes a hero,[Elizabeth Boyle],Rebecca will have to take a page from her own ...,en,[9780060549305],"[Romance, Fiction, General, History, Social Li...",2004,3,0,bookcrossing,unique
OL1000064W,One Night of Passion,NaN,One Night of Passion,[Elizabeth Boyle],They Met At London's Notorious Cyprian's Ball....,en,[9780786249909],"[Romance, Fiction, General, History, Social Li...",2002,16,0,bookcrossing,unique
...,...,...,...,...,...,...,...,...,...,...,...,...,...
unmapped_0820324019,The Unabridged Devil's Dictionary,NaN,The Unabridged Devil's Dictionary,[Ambrose Bierce],If we could only put aside our civil pose and ...,en,[9780820324012],"[Humor, Nonfiction, Classics, Reference, Langu...",1906,1,9087,goodreads,unique
unmapped_155709103X,George Washington's Rules of Civility and Dece...,NaN,George Washington's Rules of Civility and Dece...,[George Washington],Copied out by hand as a young man aspiring to ...,NaN,[9781557091031],[History],1989,1,0,google7k,unique
unmapped_1568821441,The Keeper's Companion,"Blasphemous Knowledge, Forbidden Secrets, and ...","The Keeper's Companion: Blasphemous Knowledge,...",[Keith Herber],"[CALL OF CTHULHU ROLEPLAYING] ""The Keeper's Co...",NaN,[9781568821443],[Games],2000,1,0,google7k,unique


In [3]:
# choose only first 10 tags and join them into string
books["tags"] = books["tags"].map(lambda x: ", ".join(list(x)[:10]))
text_to_embed = (
    books["title_and_subtitle"] + ". "
    + books["tags"] + ". "
    + books["description"]
)
text_to_embed

work_id
OL100003W              Home fires. Arson, Detective And Mystery Stori...
OL1000053W             No Marriage of Convenience (Avon Romantic Trea...
OL1000055W             Once Tempted. Spain, Romance, Fiction, General...
OL1000062W             It takes a hero. Romance, Fiction, General, Hi...
OL1000064W             One Night of Passion. Romance, Fiction, Genera...
                                             ...                        
unmapped_0820324019    The Unabridged Devil's Dictionary. Humor, Nonf...
unmapped_155709103X    George Washington's Rules of Civility and Dece...
unmapped_1568821441    The Keeper's Companion: Blasphemous Knowledge,...
unmapped_1569716110    Domu: A Child's Dream. Manga, Comics, Graphic ...
unmapped_188364254x    Little Novels of Sicily. Short Stories, Classi...
Length: 102117, dtype: str

In [6]:
ids = books.index.to_numpy()
position = {work_id: i for i, work_id in enumerate(ids)}

In [4]:
import time
from pathlib import Path

from chromadb.utils.embedding_functions import DefaultEmbeddingFunction  # try out different embedding functions

embed = DefaultEmbeddingFunction()   # all-MiniLM-L6-v2 (ONNX), 384 dimensions

EMB_PATH = Path("../data/embeddings_minilm.npy")
IDS_PATH = Path("../data/embeddings_minilm_ids.npy")


def embed_texts(texts: list[str], batch_size: int = 1000) -> np.ndarray:
    """Embed texts in batches, printing progress. Returns a (len(texts), 384) float32 matrix."""
    chunks = []
    start = time.perf_counter()
    for i in range(0, len(texts), batch_size):
        chunks.append(np.asarray(embed(texts[i:i + batch_size]), dtype=np.float32))
        if (i // batch_size) % 10 == 0:
            done = min(i + batch_size, len(texts))
            print(f"{done:,}/{len(texts):,}  {time.perf_counter() - start:.0f}s")
    return np.vstack(chunks)

In [5]:
sample = embed_texts(text_to_embed.iloc[:1000].tolist())
print(sample.shape, np.linalg.norm(sample, axis=1)[:5])

/home/mysa/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:08<00:00, 9.65MiB/s]


1,000/1,000  47s
(1000, 384) [1.         1.         0.99999994 1.         1.        ]


#### Full sample embedding by batches

In [8]:
def embed_full():
    if EMB_PATH.exists():
        E = np.load(EMB_PATH)
        cached_ids = np.load(IDS_PATH)
        assert len(cached_ids) == len(ids) and (cached_ids == ids.astype(str)).all(), \
            "cached embeddings belong to a different book list: delete the .npy files and rebuild"
    else:
        E = embed_texts(text_to_embed.tolist())
        np.save(EMB_PATH, E)
        np.save(IDS_PATH, ids.astype(str))
    print(E.shape)
    return E


In [9]:
E = embed_full()

1,000/102,117  42s
11,000/102,117  337s
21,000/102,117  536s
31,000/102,117  747s
41,000/102,117  963s
51,000/102,117  1248s
61,000/102,117  1537s
71,000/102,117  1814s
81,000/102,117  2125s
91,000/102,117  2413s
101,000/102,117  2699s
(102117, 384)


In [14]:
def search(query: str, k: int = 10) -> pd.DataFrame:
    """Books whose text is closest in meaning to a free-text query."""
    q = np.asarray(embed([query])[0], dtype=np.float32)   # same model as the books -> same space
    scores = E @ q                                         # cosine similarity to every book
    top = np.argsort(-scores)[:k]

    result = books.iloc[top][["title", "authors", "tags", "description"]].copy()
    #result["tags"] = result["tags"].map(lambda t: list(t)[:5])
    result.insert(0, "score", scores[top].round(3))
    return result


search("introduction to machine learning for beginners")

,score,title,authors,tags,description
work_id,,,,,
OL16036011W,0.549,The beginner's guide to computers,[Robin Bradbeer],,"Explains the development of modern computers, ..."
OL2640321W,0.496,Elements of ML programming,[Jeffrey D. Ullman],,This highly accessible introduction to the fun...
gr:30039826-the-beginner-s-photography-guide,0.462,The Beginner's Photography Guide: The Ultimate...,[Chris Gatcum],"Photography, Reference, Nonfiction","If you're new to photography, then The Beginne..."
OL6671443W,0.402,Write Your Own Adventure Programs,[Jenny Tyler],"Computers, Computer Game, Microcomputers, Prog...","Usborne Computer Books are colourful, straight..."
OL1849825W,0.393,Beginning chess,[Bruce Pandolfini],Chess,"This book ""Beginning Chess"" will help you on y..."
OL18147181W,0.392,The beginner's computer dictionary,[Elizabeth S. Wall],"Computers, Dictionaries",An alphabetically-arranged list of computer te...
OL2241588W,0.390,The personal computer book,[Peter McWilliams],Microcomputers,Contains the best from the author's previously...
gr:13025430-how-to-build-a-computer,0.389,How To Build A Computer (For Beginners),[John Gower III],"Computers, Technology",Have you ever wanted to know how to build a co...
OL58316W,0.387,Beginning programming for dummies,[Wallace Wang],"Juvenile Fiction, Computer Programs, Computer ...",Discover the latest programming tips and techn...


## Hybrid search with RRF

In [16]:
# TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words="english", min_df=3, max_df=0.5, sublinear_tf=True, dtype=np.float32)
X = vectorizer.fit_transform(text_to_embed)

In [18]:
from catalog.normalize import normalize_title
from catalog.normalize import author_key

main_author = books["authors"].map(lambda a: author_key(a[0]) if len(a) else None).to_numpy()

def source_percentile(counts: pd.Series) -> np.ndarray:
    """0-1 popularity percentile among books that have this source; 0 where count is 0."""
    has = (counts > 0).to_numpy()
    pct = np.zeros(len(counts), dtype=np.float32)
    pct[has] = counts[has].rank(pct=True).to_numpy()
    return pct


# best percentile across sources: top 10% on Goodreads or on Book-Crossing both count as popular
popularity = np.maximum(source_percentile(books["gr_popularity"]), source_percentile(books["bx_popularity"]))

# same normalized title + main author = same book (catches duplicate copies like the two Hobbits)
dedup_key = (books["title"].map(normalize_title).fillna("") + "|"
             + pd.Series(main_author, index=books.index).fillna("")).to_numpy()

assert len(popularity) == len(dedup_key) == X.shape[0] == E.shape[0]

In [19]:
from scipy import sparse

def similarity_to(vectors, pos: int) -> np.ndarray:
    """Cosine similarity of every row to row `pos` (rows must be L2-normalised)."""
    scores = vectors @ vectors[pos].T
    return scores.toarray().ravel() if sparse.issparse(scores) else np.asarray(scores).ravel()

In [20]:
def rank_positions(scores: np.ndarray) -> np.ndarray:
    """Rank of every item, 1 = highest score."""
    order = np.argsort(-scores)
    ranks = np.empty(len(scores), dtype=np.int64)
    ranks[order] = np.arange(1, len(scores) + 1)
    return ranks


def hybrid_scores(pos: int, alpha: float = 0.5, rrf_k: int = 60) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Weighted Reciprocal Rank Fusion of TF-IDF and embedding similarity to book `pos`.

    alpha: weight of TF-IDF (1.0 = TF-IDF only, 0.0 = embeddings only).
    Returns (hybrid score, TF-IDF rank, embedding rank) for every book.
    """
    tfidf_rank = rank_positions(similarity_to(X, pos))
    emb_rank = rank_positions(similarity_to(E, pos))
    hybrid = alpha / (rrf_k + tfidf_rank) + (1 - alpha) / (rrf_k + emb_rank)
    return hybrid, tfidf_rank, emb_rank

In [21]:
# hybrid = α / (60 + rank_tfidf) + (1 − α) / (60 + rank_embedding)   = Reciprocal Rank Fusion (RRF)
# final = (1 − β) · similarity + β · popularity   = rank documents by similarity to query and popularity


def recommend_hybrid(
    work_id: str,
    k: int = 10,
    alpha: float = 0.5,
    beta: float = 0.2,
    n_candidates: int = 200,
    max_query_author: int | None = 0,
    max_per_author: int | None = 2,
) -> pd.DataFrame:
    """Retrieve with hybrid TF-IDF + embedding similarity, re-rank with popularity, apply rules."""
    if work_id not in position:
        raise KeyError(f"{work_id} is not in the index (no usable description?)")
    pos = position[work_id]

    # ---- stage 1: retrieval ----
    hybrid, tfidf_rank, emb_rank = hybrid_scores(pos, alpha)
    hybrid[pos] = -np.inf                                        # never the query itself
    candidates = np.argpartition(-hybrid, n_candidates)[:n_candidates]

    # ---- stage 2: ranking ----
    sim = hybrid[candidates]
    sim = (sim - sim.min()) / (sim.max() - sim.min() + 1e-12)    # rescale to 0-1 within candidates
    final = (1 - beta) * sim + beta * popularity[candidates]
    order = np.argsort(-final)
    candidates, sim, final = candidates[order], sim[order], final[order]

    # ---- stage 3: business rules ----
    query_author = main_author[pos]
    picked, per_author = [], {}
    seen = {dedup_key[pos]}                                      # also hides other copies of the query book
    for i in range(len(candidates)):
        book = candidates[i]
        if len(picked) == k:
            break
        if dedup_key[book] in seen:
            continue
        author = main_author[book]
        if author is not None:
            cap = max_query_author if author == query_author else max_per_author
            if cap is not None and per_author.get(author, 0) >= cap:
                continue
            per_author[author] = per_author.get(author, 0) + 1
        seen.add(dedup_key[book])
        picked.append(i)

    chosen = candidates[picked]
    result = books.iloc[chosen][["title", "authors"]].copy()
    result.insert(0, "final", final[picked].round(3))
    result["similarity"] = sim[picked].round(3)
    result["popularity"] = popularity[chosen].round(2)
    result["tfidf_rank"] = tfidf_rank[chosen]
    result["emb_rank"] = emb_rank[chosen]
    return result

In [22]:
fellowship = "OL27513W"
recommend_hybrid(fellowship, alpha=1.0, beta=0.0)   # TF-IDF only, no popularity


,final,title,authors,similarity,popularity,tfidf_rank,emb_rank
work_id,,,,,,,
g7k:9780739409558,0.979,The Lord of the Rings,[],0.979,0.00,3,6
OL4488062W,0.834,The Atlas of Middle-earth,[Karen Wynn Fonstad],0.834,0.56,11,25
OL219602W,0.787,The Hobbit: Graphic Novel,[Chuck Dixon],0.787,0.98,14,8
OL5099598W,0.731,J.R.R. Tolkien's Hobbit and Lord of the rings,[Anne Pienciak],0.731,0.18,18,23
OL6214834W,0.718,Tolkien's Ordinary Virtues,[Mark Eddy Smith],0.718,0.18,19,22
g7k:9780618391004,0.705,The Lord of the Rings,[Chris Smith],0.705,0.00,20,29
OL16060879W,0.692,Bored of the Rings: A Parody of J.R.R. Tolkien...,"[The Harvard Lampoon, Henry N. Beard, Douglas ...",0.692,0.96,21,13
OL2660323W,0.668,The Raven Ring,[Patricia C. Wrede],0.668,0.56,23,209
OL15843391W,0.656,The Illearth War,[Stephen R. Donaldson],0.656,0.97,24,233


In [23]:
recommend_hybrid(fellowship, alpha=0.0, beta=0.0)   # embeddings only


,final,title,authors,similarity,popularity,tfidf_rank,emb_rank
work_id,,,,,,,
g7k:9780739409558,0.921,The Lord of the Rings,[],0.921,0.00,3,6
OL219602W,0.884,The Hobbit: Graphic Novel,[Chuck Dixon],0.884,0.98,14,8
OL16456308W,0.850,The Lord of the Rings and Philosophy,"[Gregory Bassham, Eric Bronson]",0.850,0.56,26,10
OL16060879W,0.802,Bored of the Rings: A Parody of J.R.R. Tolkien...,"[The Harvard Lampoon, Henry N. Beard, Douglas ...",0.802,0.96,21,13
OL510427W,0.758,Ringworld's Children,[Larry Niven],0.758,0.69,6367,16
OL5587819W,0.745,J.R.R. Tolkien,[Tom Shippey],0.745,0.64,44,17
g7k:9780618422531,0.731,Understanding the Lord of the Rings,"[Neil D. Isaacs, Rose A. Zimbardo]",0.731,0.00,1556,18
OL5747799W,0.718,The Lord of the rings,[David Brawn],0.718,0.18,35,19
OL6214834W,0.680,Tolkien's Ordinary Virtues,[Mark Eddy Smith],0.680,0.18,19,22


In [24]:
recommend_hybrid(fellowship, alpha=0.5, beta=0.0)   # hybrid


,final,title,authors,similarity,popularity,tfidf_rank,emb_rank
work_id,,,,,,,
g7k:9780739409558,0.952,The Lord of the Rings,[],0.952,0.00,3,6
OL219602W,0.844,The Hobbit: Graphic Novel,[Chuck Dixon],0.844,0.98,14,8
OL16060879W,0.760,Bored of the Rings: A Parody of J.R.R. Tolkien...,"[The Harvard Lampoon, Henry N. Beard, Douglas ...",0.760,0.96,21,13
OL16456308W,0.755,The Lord of the Rings and Philosophy,"[Gregory Bassham, Eric Bronson]",0.755,0.56,26,10
OL4488062W,0.753,The Atlas of Middle-earth,[Karen Wynn Fonstad],0.753,0.56,11,25
OL5099598W,0.715,J.R.R. Tolkien's Hobbit and Lord of the rings,[Anne Pienciak],0.715,0.18,18,23
OL6214834W,0.715,Tolkien's Ordinary Virtues,[Mark Eddy Smith],0.715,0.18,19,22
g7k:9780618391004,0.671,The Lord of the Rings,[Chris Smith],0.671,0.00,20,29
OL5747799W,0.650,The Lord of the rings,[David Brawn],0.650,0.18,35,19


In [25]:
recommend_hybrid(fellowship, alpha=0.5, beta=0.3)   # hybrid + popularity

,final,title,authors,similarity,popularity,tfidf_rank,emb_rank
work_id,,,,,,,
OL219602W,0.885,The Hobbit: Graphic Novel,[Chuck Dixon],0.844,0.98,14,8
OL16060879W,0.820,Bored of the Rings: A Parody of J.R.R. Tolkien...,"[The Harvard Lampoon, Henry N. Beard, Douglas ...",0.760,0.96,21,13
OL16456308W,0.698,The Lord of the Rings and Philosophy,"[Gregory Bassham, Eric Bronson]",0.755,0.56,26,10
OL4488062W,0.696,The Atlas of Middle-earth,[Karen Wynn Fonstad],0.753,0.56,11,25
g7k:9780739409558,0.667,The Lord of the Rings,[],0.952,0.00,3,6
OL6035545W,0.637,The Tolkien Companion,"[J. E. A. Tyler, John Ronald Reuel Tolkien]",0.564,0.81,29,43
OL5587819W,0.633,J.R.R. Tolkien,[Tom Shippey],0.628,0.64,44,17
OL5753836W,0.613,The Lord of the Rings,[Jude Fisher],0.600,0.64,28,35
OL5751219W,0.579,The Magical Worlds of the Lord of the Rings,[David Colbert],0.551,0.64,55,24
